# This file is dedictated to Mental Health Zero shot prompt testing and verifying Accuracy with T5 pretrained base model 

In [ ]:
# -*- coding: utf-8 -*-
"""
T5 Zero-Shot Evaluation for Drug Review Dataset
Evaluates a pre-trained T5 model (without fine-tuning) on drug review classification
"""

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import os
from tqdm import tqdm

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# CONFIGURATION

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Detect environment and set paths
if os.getenv("COLAB_RELEASE_TAG"):
    from google.colab import drive
    drive.mount('/content/gdrive/')
    colab_dir = "/content/gdrive/MyDrive/Colab Notebooks/"
    base_dir = colab_dir + "d266/FinalProject/"

    print("Running in Google Colab")
else:
    base_dir = "./"
    print("Running locally")
subfolder = "mentalhealth/"
# Model configuration
MODEL_NAME = "t5-base"  # Pre-trained T5 base model (no fine-tuning)
print(f"\nModel: {MODEL_NAME}")
print("Mode: ZERO-SHOT (no fine-tuning)")

Using device: cuda
Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
Running in Google Colab

Model: t5-base
Mode: ZERO-SHOT (no fine-tuning)


In [ ]:
# LOAD DATA

print("\n" + "="*70)
print("LOADING DATA")
print("="*70)
load_data = "CSV" # options : ['HF','CSV']
if load_data == "HF":

    try:
        from datasets import load_dataset
        print("\nLoading from Hugging Face dataset...")
        ds = load_dataset("Mouwiya/drug-reviews")

        # Convert to pandas for easier manipulation
        test_df = ds['test'].to_pandas()
        val_df = ds['validation'].to_pandas() if 'validation' in ds else ds['test'].to_pandas()

        # Rename columns if needed
        if 'review' in test_df.columns:
            test_df = test_df.rename(columns={'review': 'text', 'rating': 'labels'})
            val_df = val_df.rename(columns={'review': 'text', 'rating': 'labels'})

        print("Correct Data loaded from Hugging Face")

    except Exception as e:
        print(f"Could not load from Hugging Face: {e}")
        print("\nLoading from CSV files...")

        # Option 2: Load from CSV files
        val_df = pd.read_csv(base_dir + 'drugreview_val.csv')
        test_df = pd.read_csv(base_dir + 'drugreview_test.csv')

        print("Correct Data loaded from CSV")
elif load_data == "CSV":
    print("\nLoading data from CSV...")

    val_csv = base_dir + subfolder+"val.csv"
    test_csv = base_dir + subfolder+"test.csv"

    # LOAD DATA
    # print("\nLoading data...")
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)

# Ensure labels are in correct format
test_df['labels'] = test_df['labels'].astype(str)
val_df['labels'] = val_df['labels'].astype(str)

print(f"\nValidation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

print(f"\nLabel distribution in validation:")
print(val_df['labels'].value_counts().sort_index())

print(f"\nLabel distribution in test:")
print(test_df['labels'].value_counts().sort_index())


LOADING DATA

Loading data from CSV...

Validation samples: 5541
Test samples: 46108

Label distribution in validation:
labels
0     940
1     409
2     523
3    1014
4    2655
Name: count, dtype: int64

Label distribution in test:
labels
0     8062
1     3383
2     4254
3     8161
4    22248
Name: count, dtype: int64


In [ ]:
val_df

,Unnamed: 0,drugName,condition,review,text,labels
0,8087,amoxicillin clavulanate,skin or soft tissue infection,this med was given as a result of a deep gouge...,amoxicillin clavulanate | skin or soft tissue ...,0
1,17037,gianvi,premenstrual dysphoric disorde,i was given this generic by my pharmacy who di...,gianvi | premenstrual dysphoric disorde | i wa...,0
2,18041,liletta,birth control,i have had the liletta iud in place for about ...,liletta | birth control | i have had the lilet...,0
3,12265,ethinyl estradiol levonorgestrel,birth control,this is my second month on lutera and it has s...,ethinyl estradiol levonorgestrel | birth contr...,2
4,22139,ethinyl estradiol norgestimate,birth control,sprintec is great for me. i have taken it for ...,ethinyl estradiol norgestimate | birth control...,4
...,...,...,...,...,...,...
5536,24932,mirena,abnormal uterine bleeding,well after i was told i needed mirena for heav...,mirena | abnormal uterine bleeding | well afte...,4
5537,13038,drospirenone ethinyl estradiol,birth control,"ive been on it for 3 months, and its pretty go...",drospirenone ethinyl estradiol | birth control...,3
5538,14916,liraglutide,"diabetes, type 2","so far so good,,,been on it for 11 days have l...","liraglutide | diabetes, type 2 | so far so goo...",4
5539,10214,ethinyl estradiol norgestimate,birth control,the first couple of months were a bit odd: i e...,ethinyl estradiol norgestimate | birth control...,4


In [ ]:
label_map = {
    "0": "1-2 stars",
    "1": "3-4 stars",
    "2": "5-6 stars",
    "3": "7-8 stars",
    "4": "9-10 stars",
  }

In [ ]:
# LOAD PRE-TRAINED MODEL (ZERO-SHOT)

print("\n" + "="*70)
print("LOADING PRE-TRAINED MODEL (NO FINE-TUNING)")
print("="*70)

print(f"\nLoading {MODEL_NAME}...")
#tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
# For Drug Review dataset
model_path = base_dir +subfolder+ "t5base_finetuned_drugreviews"
print(f"\nLoading model from: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
model.eval()

print("Correct Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


LOADING PRE-TRAINED MODEL (NO FINE-TUNING)

Loading t5-base...

Loading model from: /content/gdrive/MyDrive/Colab Notebooks/d266/FinalProject/drugreview/t5base_finetuned_drugreviews
Correct Model loaded successfully!
Model parameters: 222,903,552


In [ ]:
# ZERO-SHOT PREDICTION FUNCTION

def zero_shot_predict(texts, labels_true, model, tokenizer, device, batch_size=8, prompt_template=None):
    """
    Perform zero-shot prediction on drug reviews

    Args:
        texts: List of review texts
        labels_true: True labels (0-4 or 1-5 depending on your dataset)
        model: Pre-trained T5 model
        tokenizer: T5 tokenizer
        device: torch device
        batch_size: Batch size for inference
        prompt_template: Template for zero-shot prompting

    Returns:
        predictions, true_labels, raw_outputs
    """

    if prompt_template is None:
        # Default zero-shot prompt
        prompt_template = f"Rate this drug review from 0 to 4 stars. Review: {texts}\nRating:"

    predictions = []
    raw_outputs = []

    print(f"\nProcessing {len(texts)} samples in batches of {batch_size}...")

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Zero-shot inference"):
            batch_end = min(i + batch_size, len(texts))
            batch_texts = texts[i:batch_end]

            # Create prompts
            prompts = [prompt_template.format(text=text[:500]) for text in batch_texts]

            # Tokenize
            inputs = tokenizer(
                prompts,
                max_length=512,
                truncation=True,
                padding=True,
                return_tensors="pt"
            ).to(device)

            # Generate predictions
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=5,
                num_beams=1,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            # Decode outputs
            batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            predictions.extend(batch_preds)
            raw_outputs.extend(batch_preds)

    return predictions, labels_true, raw_outputs


In [ ]:
# PARSE PREDICTIONS

def parse_predictions(raw_predictions, valid_labels=[0, 1, 2, 3, 4]):
    """
    Parse zero-shot predictions to extract numeric ratings
    Handles various formats like "4", "4 stars", "rating: 3", etc.
    """
    parsed = []

    for pred in raw_predictions:
        pred_clean = pred.strip().lower()

        # Try to find a digit
        found_digit = None
        for char in pred_clean:
            if char.isdigit():
                digit = int(char)
                if digit in valid_labels:
                    found_digit = digit
                    break

        if found_digit is not None:
            parsed.append(found_digit)
        else:
            # If no valid digit found, use a default (e.g., middle rating)
            parsed.append(-1)  # Mark as invalid

    return parsed


In [ ]:
# EVALUATE FUNCTION

def evaluate_zero_shot(df, dataset_name, model, tokenizer, device, batch_size=8, prompt_variants=None):
    """
    Evaluate zero-shot performance with multiple prompt variants
    """
    print("\n" + "="*70)
    print(f"ZERO-SHOT EVALUATION - {dataset_name.upper()} SET")
    print("="*70)

    texts = df['text'].tolist()
    labels_true = df['labels'].tolist()

    if prompt_variants is None:
        # Try multiple prompt formats
        prompt_variants = [
            f"Rate this drug review from 0 to 4. Review: {texts}\nRating:",
            f"Classify this drug review rating (0-4): {texts}\nRating:",
            f"What rating (0-4) does this drug review deserve? {texts}\nAnswer:",
            f"Review: {texts}\nRate from 0 (worst) to 4 (best):",
        ]

    best_accuracy = 0
    best_results = None
    best_prompt = None

    for idx, prompt in enumerate(prompt_variants):
        print(f"\n{'─'*70}")
        print(f"Testing Prompt {idx+1}/{len(prompt_variants)}")
        print(f"Template: {prompt[:80]}...")
        print(f"{'─'*70}")

        # Get predictions
        raw_preds, true_labels, raw_outputs = zero_shot_predict(
            texts, labels_true, model, tokenizer, device, batch_size, prompt
        )

        # Parse predictions
        parsed_preds = parse_predictions(raw_preds)

        # Show sample outputs
        print("\nSample predictions (first 10):")
        print(f"{'Index':<6} {'True':<6} {'Raw Output':<30} {'Parsed':<8} {'Status'}")
        print("-" * 70)
        for i in range(min(10, len(parsed_preds))):
            status = "Correct" if parsed_preds[i] == true_labels[i] else "Incorrect"
            print(f"{i:<6} {true_labels[i]:<6} {raw_outputs[i][:28]:<30} {parsed_preds[i]:<8} {status}")

        # Filter valid predictions
        valid_indices = [i for i, p in enumerate(parsed_preds) if p != -1]
        valid_preds = [parsed_preds[i] for i in valid_indices]
        valid_true = [true_labels[i] for i in valid_indices]

        valid_pct = len(valid_indices) / len(parsed_preds) * 100
        print(f"\nValid predictions: {len(valid_indices)}/{len(parsed_preds)} ({valid_pct:.1f}%)")

        if len(valid_preds) == 0:
            print(" No valid predictions for this prompt. Skipping...")
            continue

        # Calculate metrics
        accuracy = accuracy_score(valid_true, valid_preds)
        f1_macro = f1_score(valid_true, valid_preds, average='macro', zero_division=0)
        f1_weighted = f1_score(valid_true, valid_preds, average='weighted', zero_division=0)
        precision = precision_score(valid_true, valid_preds, average='macro', zero_division=0)
        recall = recall_score(valid_true, valid_preds, average='macro', zero_division=0)

        print(f"\nMetrics for this prompt:")
        print(f"  Accuracy:     {accuracy:.4f}")
        print(f"  F1 (Macro):   {f1_macro:.4f}")
        print(f"  F1 (Weighted): {f1_weighted:.4f}")
        print(f"  Precision:    {precision:.4f}")
        print(f"  Recall:       {recall:.4f}")

        # Track best prompt
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_prompt = prompt
            best_results = {
                'predictions': valid_preds,
                'true_labels': valid_true,
                'raw_outputs': raw_outputs,
                'accuracy': accuracy,
                'f1_macro': f1_macro,
                'f1_weighted': f1_weighted,
                'precision': precision,
                'recall': recall,
                'valid_indices': valid_indices
            }

    # Display best results
    print("\n" + "="*70)
    print("BEST PROMPT RESULTS")
    print("="*70)
    print(f"\nBest prompt: {best_prompt[:100]}...")
    print(f"\n Best Metrics:")
    print(f"  Accuracy:      {best_results['accuracy']:.4f}")
    print(f"  F1 (Macro):    {best_results['f1_macro']:.4f}")
    print(f"  F1 (Weighted): {best_results['f1_weighted']:.4f}")
    print(f"  Precision:     {best_results['precision']:.4f}")
    print(f"  Recall:        {best_results['recall']:.4f}")

    # Classification report
    print("\n" + "="*70)
    print("CLASSIFICATION REPORT (BEST PROMPT)")
    print("="*70)

    labels_range = sorted(set(best_results['true_labels']))
    report = classification_report(
        best_results['true_labels'],
        best_results['predictions'],
        labels=labels_range,
        target_names=[f"Rating {i}" for i in labels_range],
        zero_division=0
    )
    print(report)

    # Confusion matrix
    print("\n" + "="*70)
    print("CONFUSION MATRIX (BEST PROMPT)")
    print("="*70)

    cm = confusion_matrix(
        best_results['true_labels'],
        best_results['predictions'],
        labels=labels_range
    )

    print("\nConfusion Matrix (rows=true, cols=predicted):")
    print(cm)

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[f"Rating {i}" for i in labels_range]
    )
    disp.plot(cmap="Blues", ax=ax, xticks_rotation=45)
    plt.title(
        f"Zero-Shot Confusion Matrix - {dataset_name}\n"
        f"Accuracy: {best_results['accuracy']:.4f} | F1: {best_results['f1_macro']:.4f}",
        fontsize=14
    )
    plt.tight_layout()

    # Save plot
    save_path = base_dir + f"zeroshot_confusion_matrix_{dataset_name.lower()}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n Confusion matrix saved: {save_path}")
    plt.show()

    # Save predictions
    results_df = pd.DataFrame({
        'Index': best_results['valid_indices'],
        'True_Label': best_results['true_labels'],
        'Predicted_Label': best_results['predictions'],
        'Raw_Output': [best_results['raw_outputs'][i] for i in best_results['valid_indices']],
        'Correct': [p == t for p, t in zip(best_results['predictions'], best_results['true_labels'])]
    })

    csv_path = base_dir + f"zeroshot_predictions_{dataset_name.lower()}.csv"
    results_df.to_csv(csv_path, index=False)
    print(f" Predictions saved: {csv_path}")

    return best_results

In [ ]:
# RUN ZERO-SHOT EVALUATION

print("\n" + "="*70)
print("STARTING ZERO-SHOT EVALUATION")
print("="*70)
print("\nNote: Zero-shot means the model has NOT been fine-tuned on this task.")
print("We're testing if the pre-trained T5 model can classify drug reviews")
print("using only natural language prompts.\n")

# Evaluate validation set
val_results = evaluate_zero_shot(
    df=val_df,
    dataset_name="Validation",
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=8
)

# Evaluate test set
test_results = evaluate_zero_shot(
    df=test_df,
    dataset_name="Test",
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=8
)



STARTING ZERO-SHOT EVALUATION

Note: Zero-shot means the model has NOT been fine-tuned on this task.
We're testing if the pre-trained T5 model can classify drug reviews
using only natural language prompts.


ZERO-SHOT EVALUATION - VALIDATION SET

──────────────────────────────────────────────────────────────────────
Testing Prompt 1/4
Template: Rate this drug review from 0 to 4. Review: ['amoxicillin clavulanate | skin or s...
──────────────────────────────────────────────────────────────────────

Processing 5541 samples in batches of 8...


Zero-shot inference:   0%|          | 0/693 [00:00<?, ?it/s]

In [ ]:
# FINAL SUMMARY

print("\n" + "="*70)
print("ZERO-SHOT EVALUATION COMPLETE!")
print("="*70)

print("\n FINAL RESULTS SUMMARY:")
print("\nValidation Set (Zero-Shot):")
print(f"  Accuracy:      {val_results['accuracy']:.4f}")
print(f"  F1 (Macro):    {val_results['f1_macro']:.4f}")
print(f"  F1 (Weighted): {val_results['f1_weighted']:.4f}")

print("\nTest Set (Zero-Shot):")
print(f"  Accuracy:      {test_results['accuracy']:.4f}")
print(f"  F1 (Macro):    {test_results['f1_macro']:.4f}")
print(f"  F1 (Weighted): {test_results['f1_weighted']:.4f}")

print(f"\nFiles saved in: {base_dir}")
print("  Correct zeroshot_confusion_matrix_validation.png")
print("  Correct zeroshot_confusion_matrix_test.png")
print("  Correct zeroshot_predictions_validation.csv")
print("  Correct zeroshot_predictions_test.csv")

print("\n💡 Run fine-tuned model to compare these zero-shot results")
print("   to see the improvement from fine-tuning!")

# End of File